In [126]:
import cv2
import glob
import os
import numpy as np
from scipy.optimize import least_squares
from pathlib import Path
from scipy.spatial.transform import Rotation
import time
from tqdm import tqdm

In [3]:
def load_images(folder):
    images = []
    filenames = []

    for filename in sorted(glob.glob(os.path.join(folder, "*.jpg"))):
        img = cv2.imread(filename)

        images.append(img)
        filenames.append(filename)

    return images, filenames

In [10]:
SQUARE_SIZE = 20.0            # mm
PATTERN = (9, 6)              # inner corners (cols, rows)
IMAGE_GLOB = "calib/*.[jJ][pP][gG]"
CHOSEN_MODEL = "k1k2"         # model to save
SAVE_RESIDUAL_PLOT = True
SAVE_UNDISTORTED_SAMPLE = True
 
MODELS = {
    "k1k2":     cv2.CALIB_ZERO_TANGENT_DIST | cv2.CALIB_FIX_K3,
    "k1k2k3":   cv2.CALIB_ZERO_TANGENT_DIST,
    "full":     0,
    "rational": cv2.CALIB_RATIONAL_MODEL,
    "k1k2_sq":  cv2.CALIB_ZERO_TANGENT_DIST | cv2.CALIB_FIX_K3 | cv2.CALIB_FIX_ASPECT_RATIO,
}

 
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:PATTERN[0], 0:PATTERN[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE
 
 
def find_corners(gray):
    ok, corners = cv2.findChessboardCornersSB(
        gray, PATTERN, flags=cv2.CALIB_CB_EXHAUSTIVE | cv2.CALIB_CB_ACCURACY)
    if ok:
        return True, corners.astype(np.float32)
 
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
    ok, corners = cv2.findChessboardCorners(gray, PATTERN, flags)
    if not ok:
        return False, None
    corners = cv2.cornerSubPix(
        gray, corners, (25, 25), (-1, -1),
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-4))
    return True, corners
 
 
objpoints, imgpoints, names = [], [], []
img_size = None
files = sorted(glob.glob(IMAGE_GLOB))
if not files:
    raise SystemExit(f"No images found for pattern: {IMAGE_GLOB}")
 
for f in files:
    gray = cv2.imread(f, cv2.IMREAD_GRAYSCALE | cv2.IMREAD_IGNORE_ORIENTATION)
    if gray is None:
        print(f"Skipped (cannot read): {f}")
        continue
    size = gray.shape[::-1]
    if img_size is None:
        img_size = size
    elif size != img_size:
        print(f"Skipped (size {size} != {img_size}): {f}")
        continue
 
    ok, corners = find_corners(gray)
    if not ok:
        print(f"Skipped (detection failed): {f}")
        continue
    objpoints.append(objp)
    imgpoints.append(corners)
    names.append(f)
 
print(f"\nDetected corners in {len(names)} / {len(files)} images, size {img_size}")
if len(names) < 10:
    raise SystemExit("Not enough valid images for a reliable calibration. Retake photos.")
 
 
# calibration
def per_image_errors(objpts, imgpts, K, dist, rvecs, tvecs):
    errs, resid = [], []
    for op, ip, rv, tv in zip(objpts, imgpts, rvecs, tvecs):
        proj, _ = cv2.projectPoints(op, rv, tv, K, dist)
        d = ip.reshape(-1, 2) - proj.reshape(-1, 2)
        errs.append(np.sqrt(np.mean(np.sum(d ** 2, axis=1))))
        resid.append(d)
    return np.array(errs), resid
 
 
def calibrate(objpts, imgpts, flags):
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpts, imgpts, img_size, None, None, flags=flags)
    errs, resid = per_image_errors(objpts, imgpts, K, dist, rvecs, tvecs)
    return rms, K, dist, rvecs, tvecs, errs, resid
 
 
# compare models on all images
print("\n=== Model comparison (all images) ===")
for name, fl in MODELS.items():
    rms, K, dist, *_ = calibrate(objpoints, imgpoints, fl)
    print(f"{name:9s} RMS={rms:.3f}px  fx/fy={K[0,0]/K[1,1]:.4f}  "
          f"c=({K[0,2]:.0f},{K[1,2]:.0f})  dist={np.round(dist.ravel()[:5], 4)}")
 
flags = MODELS[CHOSEN_MODEL]
rms, K, dist, rvecs, tvecs, errs, resid = calibrate(objpoints, imgpoints, flags)
print(f"\n=== Chosen model: {CHOSEN_MODEL} ===")
print(f"Initial RMS {rms:.3f}px with {len(names)} images")
 
med = np.median(errs)
mad = np.median(np.abs(errs - med)) + 1e-9
thresh = med + 3 * 1.4826 * mad
bad = [i for i in np.argsort(-errs) if errs[i] > thresh][: max(1, len(names) // 5)]
if bad and len(names) - len(bad) >= 10:
    for i in bad:
        print(f"  dropping {names[i]} ({errs[i]:.2f}px > {thresh:.2f}px)")
    keep = [i for i in range(len(names)) if i not in bad]
    objpoints = [objpoints[i] for i in keep]
    imgpoints = [imgpoints[i] for i in keep]
    names = [names[i] for i in keep]
    rms, K, dist, rvecs, tvecs, errs, resid = calibrate(objpoints, imgpoints, flags)
else:
    print(" no clear outliers, keeping all images")
 
print(f"\nFinal RMS {rms:.3f}px with {len(names)} images")
print("Per-image error:")
for n, e in sorted(zip(names, errs), key=lambda x: -x[1]):
    print(f"  {e:5.2f}px  {n}")
print("K =\n", K)
print("dist =", dist.ravel())
print(f"fx/fy = {K[0,0]/K[1,1]:.4f}")
 
np.savez("calibration.npz", K=K, dist=dist, img_size=img_size, rms=rms)
np.save("K.npy", K)
np.save("dist.npy", dist)


Detected corners in 20 / 20 images, size (6000, 4000)

=== Model comparison (all images) ===
k1k2      RMS=0.604px  fx/fy=1.0073  c=(2998,1964)  dist=[-0.0698  0.049   0.      0.      0.    ]
k1k2k3    RMS=0.604px  fx/fy=1.0073  c=(2998,1964)  dist=[-0.0695  0.0474  0.      0.      0.0032]
full      RMS=0.468px  fx/fy=1.0060  c=(3032,1842)  dist=[-0.0565  0.0367 -0.0085  0.0022  0.0114]
rational  RMS=0.463px  fx/fy=1.0062  c=(3029,1842)  dist=[ 1.973821e+02 -3.169767e+02 -8.500000e-03  2.100000e-03 -8.290948e+02]
k1k2_sq   RMS=0.717px  fx/fy=1.0000  c=(2994,2020)  dist=[-0.0547  0.0227  0.      0.      0.    ]

=== Chosen model: k1k2 ===
Initial RMS 0.604px with 20 images
 no clear outliers, keeping all images

Final RMS 0.604px with 20 images
Per-image error:
   0.69px  calib\DSC08078.JPG
   0.67px  calib\DSC08048.JPG
   0.66px  calib\DSC08059.JPG
   0.66px  calib\DSC08073.JPG
   0.63px  calib\DSC08075.JPG
   0.63px  calib\DSC08081.JPG
   0.62px  calib\DSC08050.JPG
   0.62px  calib\D

In [107]:
K = np.load("K.npy"); dist = np.load("dist.npy")
images, filenames = load_images("images")
images = [cv2.undistort(im, K, dist) for im in images]   # then compute SIFT
dist = np.zeros((5, 1)) 

sift = cv2.SIFT_create(8000)
keypoints = {}
descriptors = {}

for image_id, image in enumerate(images):

    k, d = sift.detectAndCompute(image, None)
    keypoints[image_id] = k
    descriptors[image_id] = d

In [110]:
# Features + matching
k0 = keypoints[0]
k1 = keypoints[1]
d0 = descriptors[0]
d1 = descriptors[1]
matches = cv2.BFMatcher().knnMatch(d0, d1, k=2)
good12 = [m for m, n in matches if m.distance < 0.75 * n.distance]  # Lowe's ratio test

p0 = np.float32([k0[m.queryIdx].pt for m in good12])
p1 = np.float32([k1[m.trainIdx].pt for m in good12])

# Relative pose from the essential matrix
E, mask = cv2.findEssentialMat(p0, p1, K, cv2.RANSAC, 0.999, 1.0)
_, R, t, mask = cv2.recoverPose(E, p0, p1, K, mask=mask)

# Triangulate
P0 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
P1 = K @ np.hstack([R, t])
inl = mask.ravel() > 0
pts4d = cv2.triangulatePoints(P0, P1, p0[inl].T, p1[inl].T)
pts3d = (pts4d[:3] / pts4d[3]).T

In [111]:

class GlobalTracks:

    def __init__(self):
        self.tracks = {}
        self.next_track_id = 0
        self.index = {}          # (camera_id, feature_id) -> track_id

    def add_matches(self, c1_id, c2_id, matches, points3d=None):

        for i, j in matches:
            t1 = self.index.get((c1_id, i))
            t2 = self.index.get((c2_id, j))

            # Neither feature is in a track -> new track
            if t1 is None and t2 is None:
                tid = self.next_track_id
                self.next_track_id += 1
                self.tracks[tid] = {c1_id: i, c2_id: j}
                self.index[(c1_id, i)] = tid
                self.index[(c2_id, j)] = tid

            # Only feature i is in a track
            elif t1 is not None and t2 is None:
                if c2_id in self.tracks[t1]:      # track already has another feature in c2
                    continue
                self.tracks[t1][c2_id] = j
                self.index[(c2_id, j)] = t1

            # Only feature j is in a track
            elif t1 is None and t2 is not None:
                if c1_id in self.tracks[t2]:
                    continue
                self.tracks[t2][c1_id] = i
                self.index[(c1_id, i)] = t2

            # Both are in tracks
            else:
                if t1 == t2:
                    continue

                a, b = self.tracks[t1], self.tracks[t2]

                # Conflict: same camera, different feature
                if any(cam in a and a[cam] != f for cam, f in b.items()):
                    continue

                a.update(b)
                for cam, f in b.items():
                    self.index[(cam, f)] = t1
                del self.tracks[t2]

                # Keep the 3D points consistent with the merge
                if points3d is not None and t2 in points3d:
                    if t1 not in points3d:
                        points3d[t1] = points3d[t2]
                    del points3d[t2]

        return self.tracks

In [112]:
global_tracks = GlobalTracks()

# Matches that survived the essential-matrix RANSAC + cheirality check.
# pts3d was triangulated from p0[inl], p1[inl], so it has the SAME order as this list.
inl = mask.ravel() > 0
initial_matches = [(m.queryIdx, m.trainIdx) for m, ok in zip(good12, inl) if ok]
assert len(initial_matches) == len(pts3d)

tracks = global_tracks.add_matches(0, 1, initial_matches)

# Assign each 3D point to the track created from ITS OWN match
points3d = {}
skipped = 0

for (i, j), X in zip(initial_matches, pts3d):
    tid = global_tracks.index.get((0, i))

    # The match was rejected by add_matches (e.g. feature j already used by another
    # match), or the track holds a different feature in image 1 -> skip it
    if tid is None or global_tracks.tracks[tid].get(1) != j:
        skipped += 1
        continue

    if not np.all(np.isfinite(X)):
        skipped += 1
        continue

    points3d[tid] = X

print("Initial matches:", len(initial_matches))
print("Tracks:", len(tracks))
print("3D points:", len(points3d), f"(skipped {skipped})")

Initial matches: 1610
Tracks: 1596
3D points: 1596 (skipped 14)


In [121]:
def find_camera_pose(tracks, points3d, img3, img2, c2_id, K, dist):

    # SIFT
    k2, d2 = sift.detectAndCompute(img2, None)
    k3, d3 = sift.detectAndCompute(img3, None)

    # Match image 2 (previous) and image 3 (new)
    matches = cv2.BFMatcher().knnMatch(d2, d3, k=2)
    good23 = [m for m, n in matches if m.distance < 0.75 * n.distance]

    # Lookup: feature index in image 2 -> track id (avoids scanning all tracks per match)
    feature_to_track = {
        track[c2_id]: track_id
        for track_id, track in tracks.items()
        if c2_id in track
    }

    object_points = []
    image_points = []
    used_matches = []      # parallel to object_points / image_points

    for m in good23:
        track_id = feature_to_track.get(m.queryIdx)

        # Only use features whose track has a triangulated 3D point
        if track_id is None or track_id not in points3d:
            continue

        object_points.append(points3d[track_id])
        image_points.append(k3[m.trainIdx].pt)
        used_matches.append(m)

    if len(object_points) < 6:
        raise RuntimeError(f"Not enough 2D-3D correspondences for PnP: {len(object_points)}")

    object_points = np.asarray(object_points, dtype=np.float32)
    image_points = np.asarray(image_points, dtype=np.float32)

    print("2D-3D correspondences:", len(object_points))

    # PnP + RANSAC
    success, rvec, tvec, inliers = cv2.solvePnPRansac(
        object_points, image_points, K, dist,
        iterationsCount=2000,
        reprojectionError=3.0,
        confidence=0.999,
        flags=cv2.SOLVEPNP_ITERATIVE,
    )

    if not success or inliers is None:
        raise RuntimeError("PnP failed")

    print("PnP inliers:", len(inliers))

    # `inliers` indexes object_points/image_points, which are aligned with
    # used_matches (NOT with good23), so index used_matches.
    good23_inliers = [used_matches[i] for i in inliers.ravel()]

    R, _ = cv2.Rodrigues(rvec)
    P = K @ np.hstack([R, tvec.reshape(3, 1)])

    return P, inliers, good23_inliers, good23

In [122]:
cameras = {0: P0, 1: P1}

In [116]:
def camera_to_params(P, K):

    Rt = np.linalg.inv(K) @ P
    R = Rt[:, :3]
    t = Rt[:, 3]
    rvec, _ = cv2.Rodrigues(R)
    return np.hstack([rvec.ravel(),t])

In [117]:
def verify_matches_fundamental(k_a, k_b, matches, thresh=1.0):
    """Keep only matches consistent with a single epipolar geometry."""
    if len(matches) < 8:
        return []

    pa = np.float32([k_a[m.queryIdx].pt for m in matches])
    pb = np.float32([k_b[m.trainIdx].pt for m in matches])

    F, mask = cv2.findFundamentalMat(pa, pb, cv2.FM_RANSAC, thresh, 0.999)
    if F is None or mask is None:
        return []

    return [m for m, ok in zip(matches, mask.ravel()) if ok]


def _pose_from_P(P, K):
    Rt = np.linalg.inv(K) @ P
    return Rt[:, :3], Rt[:, 3]


def triangulate_new_tracks(tracks, points3d, cameras, keypoints,
                           cam_a, cam_b, K,
                           max_reproj_err=2.0, min_angle_deg=1.0):
    """Triangulate tracks seen in cam_a and cam_b that have no 3D point yet."""

    P_a, P_b = cameras[cam_a], cameras[cam_b]

    ids, pts_a, pts_b = [], [], []
    for tid, track in tracks.items():
        if tid in points3d or cam_a not in track or cam_b not in track:
            continue
        ids.append(tid)
        pts_a.append(keypoints[cam_a][track[cam_a]].pt)
        pts_b.append(keypoints[cam_b][track[cam_b]].pt)

    if not ids:
        return 0

    pts_a = np.asarray(pts_a, dtype=np.float64)     # (N, 2)
    pts_b = np.asarray(pts_b, dtype=np.float64)

    X4 = cv2.triangulatePoints(P_a, P_b, pts_a.T, pts_b.T)
    X = (X4[:3] / X4[3]).T                          # (N, 3)
    Xh = np.hstack([X, np.ones((len(X), 1))])

    valid = np.isfinite(X).all(axis=1)

    # Positive depth and small reprojection error in both cameras
    for P, pts in ((P_a, pts_a), (P_b, pts_b)):
        proj = Xh @ P.T
        depth = proj[:, 2]
        with np.errstate(divide="ignore", invalid="ignore"):
            uv = proj[:, :2] / depth[:, None]
        err = np.linalg.norm(uv - pts, axis=1)
        valid &= (depth > 0) & (err < max_reproj_err)

    # Enough triangulation angle (rejects near-parallel rays -> unstable depth)
    R_a, t_a = _pose_from_P(P_a, K)
    R_b, t_b = _pose_from_P(P_b, K)
    C_a = -R_a.T @ t_a
    C_b = -R_b.T @ t_b

    ra = X - C_a
    rb = X - C_b
    with np.errstate(divide="ignore", invalid="ignore"):
        cos = np.sum(ra * rb, axis=1) / (np.linalg.norm(ra, axis=1) * np.linalg.norm(rb, axis=1))
    angle = np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
    valid &= np.nan_to_num(angle) > min_angle_deg

    added = 0
    for tid, x, ok in zip(ids, X, valid):
        if ok:
            points3d[tid] = x
            added += 1

    return added

In [ ]:
class BundleAdjustment:

    def __init__(self, cameras, points3d, tracks, keypoints, K, dist):

        self.cameras = cameras
        self.points3d = points3d
        self.tracks = tracks
        self.keypoints = keypoints
        self.K = K
        self.dist = np.asarray(dist, dtype=np.float64).reshape(-1, 1)

        # Camera 0 is fixed
        self.camera_ids = list(cameras.keys())
        self.optimized_camera_ids = [cid for cid in self.camera_ids if cid != 0]
        self.point_ids = list(points3d.keys())

        # Map IDs -> index
        self.camera_index = {camera_id: i for i, camera_id in enumerate(self.optimized_camera_ids)}

        self.point_index = {track_id: i for i, track_id in enumerate(self.point_ids)}

        # Build observations
        self.observations = self._build_observations()

        obs = self.observations
        self.obs_pt = np.array([self.point_index[t] for _, t, _, _ in obs])
        self.obs_cam_idx = np.array([self.camera_index.get(c, -1) for c, _, _, _ in obs])  # -1 = camera 0
        self.obs_uv = np.array([[u, v] for _, _, u, v in obs], dtype=np.float64)


    def _build_observations(self):

        observations = []

        for track_id in self.point_ids:

            track = self.tracks[track_id]

            for camera_id, feature_id in track.items():

                # Only use cameras that exist
                if camera_id not in self.cameras:
                    continue

                # Get measured pixel location
                u, v = self.keypoints[camera_id][feature_id].pt

                observations.append((camera_id, track_id, u, v))

        return observations


    def _camera_to_params(self, P):

        # P = K [R | t]

        Rt = np.linalg.inv(self.K) @ P

        R = Rt[:, :3]
        t = Rt[:, 3]

        rvec, _ = cv2.Rodrigues(R)

        return np.hstack([rvec.ravel(), t])


    def _create_initial_vector(self):

        x0 = []

        # Only optimize cameras 1,2,3,...
        for camera_id in self.optimized_camera_ids:
            params = self._camera_to_params(self.cameras[camera_id])
            x0.extend(params)

        # Optimize all 3D points
        for track_id in self.point_ids:
            x0.extend(self.points3d[track_id])

        return np.asarray(x0, dtype=np.float64)


    def _unpack_parameters(self, params):

        num_cameras = len(self.optimized_camera_ids)

        num_points = len(self.point_ids)

        camera_params = params[:6 * num_cameras]

        point_params = params[6 * num_cameras:]

        camera_params = camera_params.reshape(num_cameras, 6)

        point_params = point_params.reshape(num_points, 3)

        return camera_params, point_params

    def _jac_sparsity(self):
        from scipy.sparse import lil_matrix
        n_obs = len(self.observations)
        n_cam = len(self.optimized_camera_ids)
        A = lil_matrix((2 * n_obs, 6 * n_cam + 3 * len(self.point_ids)), dtype=int)

        rows = np.arange(n_obs)
        m = self.obs_cam_idx >= 0
        for s in range(6):
            A[2 * rows[m],     6 * self.obs_cam_idx[m] + s] = 1
            A[2 * rows[m] + 1, 6 * self.obs_cam_idx[m] + s] = 1
        for s in range(3):
            A[2 * rows,     6 * n_cam + 3 * self.obs_pt + s] = 1
            A[2 * rows + 1, 6 * n_cam + 3 * self.obs_pt + s] = 1
        return A


    def _residuals(self, params):
        cam_p, pts = self._unpack_parameters(params)

        X = pts[self.obs_pt]                                   # (N, 3)
        rvec = np.zeros((len(X), 3))
        tvec = np.zeros((len(X), 3))
        m = self.obs_cam_idx >= 0
        rvec[m] = cam_p[self.obs_cam_idx[m], :3]
        tvec[m] = cam_p[self.obs_cam_idx[m], 3:]

        # Rodrigues rotation, vectorized
        theta = np.linalg.norm(rvec, axis=1, keepdims=True)
        with np.errstate(invalid="ignore", divide="ignore"):
            k = np.where(theta > 1e-12, rvec / theta, 0.0)
        c, s = np.cos(theta), np.sin(theta)
        Xr = X * c + np.cross(k, X) * s + k * np.sum(k * X, axis=1, keepdims=True) * (1 - c)

        Xc = Xr + tvec
        K = self.K
        u = K[0, 0] * Xc[:, 0] / Xc[:, 2] + K[0, 2]
        v = K[1, 1] * Xc[:, 1] / Xc[:, 2] + K[1, 2]

        return (np.column_stack([u, v]) - self.obs_uv).ravel()


    def optimize(self):

        print("Starting bundle adjustment...")

        # Initial parameter vector
        x0 = self._create_initial_vector()
        print("Number of cameras:", len(self.camera_ids))
        print("Number of 3D points:", len(self.point_ids))
        print("Number of observations:", len(self.observations))

        # Initial reprojection error
        initial_residuals = self._residuals(x0)
        initial_rmse = np.sqrt(np.mean(initial_residuals ** 2))
        print("Initial reprojection RMSE:", initial_rmse)

        # Build the sparsity pattern (can take a while for big problems)
        t0 = time.time()
        sparsity = self._jac_sparsity()
        print(f"Jacobian sparsity built in {time.time() - t0:.1f}s")

        # Progress bar: wraps the residual function and updates on every call
        pbar = tqdm(
        desc="Bundle adjustment",
        unit=" evals",
        bar_format="{desc}: {n_fmt} evals | elapsed {elapsed} | {rate_fmt} | {postfix}",
        )
        best = {"rmse": initial_rmse}

        def residuals_with_progress(params):
            r = self._residuals(params)
            rmse = np.sqrt(np.mean(r ** 2))
            best["rmse"] = min(best["rmse"], rmse)
            pbar.set_postfix(rmse=f"{rmse:.4f}", best=f"{best['rmse']:.4f}")
            pbar.update(1)
            return r

        # Run optimization
        start = time.time()
        try:
            r = initial_residuals.reshape(-1, 2)
            err = np.linalg.norm(r, axis=1)
            print("median err px:", np.median(err), "| 95%:", np.percentile(err, 95), "| max:", err.max())
            print("obs with err > 50px:", np.sum(err > 50))
            result = least_squares(self._residuals, x0, method="trf", verbose=2,
                       jac_sparsity=sparsity, loss="huber", f_scale=2.0,
                       x_scale="jac", max_nfev=30, ftol=1e-6, xtol=1e-6, gtol=1e-6)
        finally:
            pbar.close()
        print(f"Optimization took {time.time() - start:.1f}s")

        print("Bundle adjustment finished.")
        print("Optimization success:", result.success)
        print("Message:", result.message)
        print("Function evaluations:", result.nfev, "| Jacobian evaluations:", result.njev)

        # Final reprojection error
        final_residuals = self._residuals(result.x)
        final_rmse = np.sqrt(np.mean(final_residuals ** 2))
        print("Final reprojection RMSE:", final_rmse)

        # Convert optimized vector back
        optimized_cameras, optimized_points3d = self._extract_results(result.x)

        return (optimized_cameras, optimized_points3d)


    def _extract_results(self, params):

        camera_params, point_params = self._unpack_parameters(params)

        optimized_cameras = {}

        # Keep camera 0 fixed
        optimized_cameras[0] = self.cameras[0]

        # Recover optimized cameras
        for i, camera_id in enumerate(self.optimized_camera_ids):

            camera = camera_params[i]

            rvec = camera[:3]
            tvec = camera[3:]

            R, _ = cv2.Rodrigues(rvec)

            P = self.K @ np.hstack([R, tvec.reshape(3, 1)])

            optimized_cameras[camera_id] = P

        # Recover optimized 3D points
        optimized_points3d = {}

        for i, track_id in enumerate(self.point_ids):
            optimized_points3d[track_id] = point_params[i]

        return (optimized_cameras, optimized_points3d)


In [128]:
import time

import cv2
import numpy as np
from scipy.optimize import least_squares
from scipy.sparse import coo_matrix
from tqdm import tqdm

try:
    # Lets us count *real* iterations (see optimize). Falls back gracefully if missing.
    from scipy.optimize._numdiff import approx_derivative
except Exception:  # pragma: no cover
    approx_derivative = None


class BundleAdjustment:
    """
    Bundle adjustment with:
      - keypoint undistortion (so the pinhole residual matches the data)
      - input cleaning (points behind cameras, large initial errors, short tracks)
      - gauge fixing: camera 0 fixed + one translation component of camera 1 fixed (scale)
      - real progress bar (percent of max_nfev, elapsed / remaining time, RMSE, median error)
    """

    def __init__(self, cameras, points3d, tracks, keypoints, K, dist,
                 min_views=3, max_init_error_px=30.0, fix_scale=True):

        self.cameras = cameras
        self.points3d = points3d
        self.tracks = tracks
        self.keypoints = keypoints
        self.K = np.asarray(K, dtype=np.float64)
        self.dist = np.asarray(dist, dtype=np.float64).reshape(-1, 1)

        self.min_views = min_views
        self.max_init_error_px = max_init_error_px
        self.fix_scale = fix_scale

        # Camera 0 is fixed (its pose is whatever it is in `cameras`, usually K[I|0])
        self.camera_ids = list(cameras.keys())
        self.optimized_camera_ids = [cid for cid in self.camera_ids if cid != 0]
        self.camera_index = {cid: i for i, cid in enumerate(self.optimized_camera_ids)}

        self.cam_params = {cid: self._camera_to_params(cameras[cid]) for cid in self.camera_ids}
        self.cam0_params = self.cam_params[0]

        # Raw observations -> undistort -> clean -> final arrays
        cam_ids, track_ids, uv = self._build_observations()
        uv = self._undistort(uv)
        self._clean(cam_ids, track_ids, uv)

        # Free / fixed parameters
        self._setup_free_parameters()

    # ------------------------------------------------------------------ setup

    def _build_observations(self):
        cam_ids, track_ids, uv = [], [], []

        for track_id in self.points3d.keys():
            track = self.tracks[track_id]
            for camera_id, feature_id in track.items():
                if camera_id not in self.cameras:
                    continue
                u, v = self.keypoints[camera_id][feature_id].pt
                cam_ids.append(camera_id)
                track_ids.append(track_id)
                uv.append((u, v))

        return (np.asarray(cam_ids), np.asarray(track_ids),
                np.asarray(uv, dtype=np.float64).reshape(-1, 2))

    def _undistort(self, uv):
        if not np.any(self.dist):
            return uv
        und = cv2.undistortPoints(uv.reshape(-1, 1, 2), self.K, self.dist, P=self.K)
        return und.reshape(-1, 2)

    def _camera_to_params(self, P):
        # P = K [R | t]
        Rt = np.linalg.inv(self.K) @ P
        R = Rt[:, :3]
        t = Rt[:, 3]

        # Make R a proper rotation (removes numerical drift / scale)
        U, _, Vt = np.linalg.svd(R)
        R = U @ Vt
        if np.linalg.det(R) < 0:
            R = -R

        rvec, _ = cv2.Rodrigues(R)
        return np.hstack([rvec.ravel(), t])

    def _clean(self, cam_ids, track_ids, uv):
        n_raw = len(uv)

        # Residuals / depths at the initial solution
        cam_p = np.array([self.cam_params[c] for c in cam_ids])
        X = np.array([self.points3d[t] for t in track_ids], dtype=np.float64)
        proj, depth = self._project(cam_p, X)
        err = np.linalg.norm(proj - uv, axis=1)

        print("Initial reprojection error (px): median %.2f | 95%% %.2f | max %.2f"
              % (np.median(err), np.percentile(err, 95), err.max()))

        keep = (depth > 0) & (err < self.max_init_error_px)
        n_bad = n_raw - int(keep.sum())

        # Drop short tracks (iterate: removing observations can shorten other tracks)
        uniq, inv = np.unique(track_ids, return_inverse=True)
        while True:
            counts = np.bincount(inv[keep], minlength=len(uniq))
            new_keep = keep & (counts[inv] >= self.min_views)
            if new_keep.sum() == keep.sum():
                break
            keep = new_keep
        counts = np.bincount(inv[keep], minlength=len(uniq))

        valid = counts >= self.min_views
        new_idx = -np.ones(len(uniq), dtype=int)
        new_idx[valid] = np.arange(valid.sum())

        self.point_ids = uniq[valid].tolist()
        self.removed_point_ids = uniq[~valid].tolist()
        self.point_index = {tid: i for i, tid in enumerate(self.point_ids)}

        self.obs_cam_id = cam_ids[keep]
        self.obs_pt = new_idx[inv[keep]]
        self.obs_cam_idx = np.array([self.camera_index.get(c, -1) for c in self.obs_cam_id], dtype=int)
        self.obs_uv = uv[keep]
        self.observations = list(zip(self.obs_cam_id, np.array(self.point_ids)[self.obs_pt],
                                     self.obs_uv[:, 0], self.obs_uv[:, 1]))

        print("Cleaning: %d -> %d observations (%d bad depth/error), %d -> %d points (min_views=%d)"
              % (n_raw, len(self.obs_uv), n_bad, len(uniq), len(self.point_ids), self.min_views))

        # Warn about cameras with no remaining observations
        seen = set(self.obs_cam_id.tolist())
        missing = [c for c in self.optimized_camera_ids if c not in seen]
        if missing:
            print("WARNING: cameras with no observations after cleaning:", missing)

    def _setup_free_parameters(self):
        n_cam = len(self.optimized_camera_ids)
        n_pts = len(self.point_ids)
        self.n_cam = n_cam
        self.n_par = 6 * n_cam + 3 * n_pts

        self.x_full0 = self._create_initial_vector()

        free = np.ones(self.n_par, dtype=bool)
        if self.fix_scale and n_cam > 0:
            # Fix the dominant translation component of the first optimized camera
            axis = int(np.argmax(np.abs(self.x_full0[3:6])))
            free[3 + axis] = False
            print("Scale fixed via camera %d, translation axis %d"
                  % (self.optimized_camera_ids[0], axis))
        self.free_idx = np.flatnonzero(free)

    # ------------------------------------------------------------ parameters

    def _create_initial_vector(self):
        cam_part = [self.cam_params[cid] for cid in self.optimized_camera_ids]
        pt_part = [np.asarray(self.points3d[tid], dtype=np.float64).ravel() for tid in self.point_ids]
        parts = cam_part + pt_part
        return np.concatenate(parts) if parts else np.zeros(0)

    def _expand(self, params_free):
        full = self.x_full0.copy()
        full[self.free_idx] = params_free
        return full

    def _unpack_parameters(self, full):
        camera_params = full[:6 * self.n_cam].reshape(self.n_cam, 6)
        point_params = full[6 * self.n_cam:].reshape(len(self.point_ids), 3)
        return camera_params, point_params

    # ------------------------------------------------------------ projection

    def _project(self, cam_p, X):
        """cam_p: (N,6) [rvec, tvec] per observation, X: (N,3). Returns (N,2) pixels, (N,) depth."""
        rvec = cam_p[:, :3]
        tvec = cam_p[:, 3:]

        # Rodrigues rotation, vectorized
        theta = np.linalg.norm(rvec, axis=1, keepdims=True)
        small = theta <= 1e-12
        k = np.where(small, 0.0, rvec / np.where(small, 1.0, theta))
        c, s = np.cos(theta), np.sin(theta)
        Xr = X * c + np.cross(k, X) * s + k * np.sum(k * X, axis=1, keepdims=True) * (1 - c)

        Xc = Xr + tvec
        z = Xc[:, 2]
        zs = np.where(np.abs(z) < 1e-9, 1e-9, z)

        K = self.K
        x = Xc[:, 0] / zs
        y = Xc[:, 1] / zs
        u = K[0, 0] * x + K[0, 1] * y + K[0, 2]
        v = K[1, 1] * y + K[1, 2]
        return np.column_stack([u, v]), z

    def _residuals(self, params_free):
        full = self._expand(params_free)
        cam_p, pts = self._unpack_parameters(full)

        X = pts[self.obs_pt]
        cam_obs = np.tile(self.cam0_params, (len(X), 1))
        m = self.obs_cam_idx >= 0
        cam_obs[m] = cam_p[self.obs_cam_idx[m]]

        proj, _ = self._project(cam_obs, X)
        return (proj - self.obs_uv).ravel()

    def _jac_sparsity(self):
        n_obs = len(self.obs_uv)
        rows = np.arange(n_obs)
        m = self.obs_cam_idx >= 0

        r_list, c_list = [], []
        for s in range(6):
            for k in (0, 1):
                r_list.append(2 * rows[m] + k)
                c_list.append(6 * self.obs_cam_idx[m] + s)
        for s in range(3):
            for k in (0, 1):
                r_list.append(2 * rows + k)
                c_list.append(6 * self.n_cam + 3 * self.obs_pt + s)

        r = np.concatenate(r_list)
        c = np.concatenate(c_list)
        A = coo_matrix((np.ones(len(r), dtype=np.int8), (r, c)),
                       shape=(2 * n_obs, self.n_par)).tocsr()
        return A[:, self.free_idx]

    # -------------------------------------------------------------- optimize

    def _run_stage(self, x0, sparsity, loss, f_scale, max_nfev, ftol, xtol, gtol, label):
        pbar = tqdm(total=max_nfev, desc=label, unit="eval",
                    bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} "
                               "[{elapsed}<{remaining}] {postfix}")
        r_init = self._residuals(x0)
        state = {"best": np.sqrt(np.mean(r_init ** 2)), "jac": 0}

        def fun(params):
            r = self._residuals(params)
            rmse = np.sqrt(np.mean(r ** 2))
            med = np.median(np.linalg.norm(r.reshape(-1, 2), axis=1))
            state["best"] = min(state["best"], rmse)
            pbar.set_postfix(iter=state["jac"], rmse=f"{rmse:.3f}",
                             best=f"{state['best']:.3f}", median=f"{med:.3f}px")
            if pbar.n < max_nfev:
                pbar.update(1)
            return r

        if approx_derivative is not None:
            # Own Jacobian callable: finite-difference calls use the raw residual
            # function, so `fun` is called once per real evaluation and the bar
            # is a true percentage of max_nfev.
            def jac(params):
                state["jac"] += 1
                return approx_derivative(self._residuals, params, method="2-point",
                                         sparsity=sparsity)
            kwargs = dict(jac=jac)
        else:
            kwargs = dict(jac="2-point", jac_sparsity=sparsity)

        start = time.time()
        try:
            result = least_squares(fun, x0, method="trf", verbose=0,
                                   loss=loss, f_scale=f_scale, x_scale="jac",
                                   max_nfev=max_nfev, ftol=ftol, xtol=xtol, gtol=gtol,
                                   **kwargs)
        finally:
            pbar.close()
        print("%s: %.1fs | nfev=%d njev=%d | %s"
              % (label, time.time() - start, result.nfev, result.njev, result.message))
        return result

    def optimize(self, max_nfev=50, ftol=1e-6, xtol=1e-6, gtol=1e-6, f_scale=2.0):
        """
        Two stages:
          1. plain least squares (loss="linear") to converge from the rough initial guess.
             A robust loss here STALLS: with initial errors >> f_scale, Huber down-weights
             almost every residual and the gradient vanishes.
          2. Huber loss, started from the stage-1 solution (residuals are now small, so
             only genuine outliers get down-weighted).
        """
        print("Starting bundle adjustment...")
        print("Number of cameras:", len(self.camera_ids))
        print("Number of 3D points:", len(self.point_ids))
        print("Number of observations:", len(self.obs_uv))
        print("Free parameters:", len(self.free_idx))

        x0 = self.x_full0[self.free_idx]

        r0 = self._residuals(x0)
        e0 = np.linalg.norm(r0.reshape(-1, 2), axis=1)
        print("Initial reprojection RMSE: %.4f | median err: %.4f px"
              % (np.sqrt(np.mean(r0 ** 2)), np.median(e0)))

        t0 = time.time()
        sparsity = self._jac_sparsity()
        print("Jacobian sparsity built in %.2fs" % (time.time() - t0))

        res1 = self._run_stage(x0, sparsity, "linear", 1.0, max_nfev,
                               ftol, xtol, gtol, "Stage 1 (linear)")
        res2 = self._run_stage(res1.x, sparsity, "huber", f_scale, max_nfev,
                               ftol, xtol, gtol, "Stage 2 (huber)")
        result = res2

        rf = self._residuals(result.x)
        ef = np.linalg.norm(rf.reshape(-1, 2), axis=1)
        print("Final reprojection RMSE: %.4f | median err: %.4f px | inliers (<2px): %.1f%%"
              % (np.sqrt(np.mean(rf ** 2)), np.median(ef), 100.0 * np.mean(ef < 2.0)))

        optimized_cameras, optimized_points3d = self._extract_results(self._expand(result.x))
        return (optimized_cameras, optimized_points3d)

    # --------------------------------------------------------------- results

    def _extract_results(self, full):
        camera_params, point_params = self._unpack_parameters(full)

        optimized_cameras = {0: self.cameras[0]}  # camera 0 stays fixed

        for i, camera_id in enumerate(self.optimized_camera_ids):
            rvec = camera_params[i, :3]
            tvec = camera_params[i, 3:]
            R, _ = cv2.Rodrigues(rvec)
            optimized_cameras[camera_id] = self.K @ np.hstack([R, tvec.reshape(3, 1)])

        # Only points that survived cleaning are returned
        optimized_points3d = {tid: point_params[i] for i, tid in enumerate(self.point_ids)}

        return (optimized_cameras, optimized_points3d)

In [119]:
def main(tracks, cameras, points3d):

    for new_camera_id in range(2, len(images)):

        print(f"Adding camera {new_camera_id}")
        previous_camera_id = new_camera_id - 1

        # 1. Pose of the new camera from existing 3D points
        P, _, pnp_inliers, all_matches = find_camera_pose(
            tracks, points3d,
            images[new_camera_id], images[previous_camera_id],
            previous_camera_id, K, dist)
        cameras[new_camera_id] = P

        # 2. Geometrically verify ALL matches between the two images
        verified = verify_matches_fundamental(
            keypoints[previous_camera_id], keypoints[new_camera_id], all_matches)

        # 3. Matches whose previous feature already has a 3D point must also be
        #    PnP inliers (otherwise a PnP outlier would be attached to a good track)
        pnp_set = {(m.queryIdx, m.trainIdx) for m in pnp_inliers}
        matches_to_add = []
        for m in verified:
            tid = global_tracks.index.get((previous_camera_id, m.queryIdx))
            if tid is not None and tid in points3d and (m.queryIdx, m.trainIdx) not in pnp_set:
                continue
            matches_to_add.append((m.queryIdx, m.trainIdx))

        global_tracks.add_matches(previous_camera_id, new_camera_id,
                                  matches_to_add, points3d)
        tracks = global_tracks.tracks

        # 4. Triangulate tracks that now see both cameras but have no 3D point
        new_points = triangulate_new_tracks(
            tracks, points3d, cameras, keypoints,
            previous_camera_id, new_camera_id, K)

        print(f"Verified matches: {len(verified)}, added to tracks: {len(matches_to_add)}")
        print(f"Added {new_points} new 3D points")
        print(f"Total tracks: {len(tracks)}")
        print(f"Total 3D points: {len(points3d)}")

    print("\nIncremental reconstruction finished.")

    # bundle adjustment
    ba = BundleAdjustment(cameras, points3d, tracks, keypoints, K, dist)
    cameras, points3d = ba.optimize()

    print("\nFinal reconstruction:")
    print("Cameras:", len(cameras))
    print("3D points:", len(points3d))

    return cameras, tracks, points3d

In [129]:
cameras, tracks, points3d = main(tracks, cameras, points3d)

Adding camera 2
2D-3D correspondences: 1370
PnP inliers: 1185
Verified matches: 1250, added to tracks: 1121
Added 1 new 3D points
Total tracks: 12196
Total 3D points: 8376
Adding camera 3
2D-3D correspondences: 883
PnP inliers: 636
Verified matches: 738, added to tracks: 568
Added 0 new 3D points
Total tracks: 12185
Total 3D points: 8365
Adding camera 4
2D-3D correspondences: 1111
PnP inliers: 957
Verified matches: 953, added to tracks: 856
Added 0 new 3D points
Total tracks: 12185
Total 3D points: 8365
Adding camera 5
2D-3D correspondences: 908
PnP inliers: 531
Verified matches: 884, added to tracks: 610
Added 6 new 3D points
Total tracks: 12185
Total 3D points: 8371
Adding camera 6
2D-3D correspondences: 287
PnP inliers: 178
Verified matches: 214, added to tracks: 158
Added 0 new 3D points
Total tracks: 12185
Total 3D points: 8371
Adding camera 7
2D-3D correspondences: 519
PnP inliers: 471
Verified matches: 484, added to tracks: 453
Added 0 new 3D points
Total tracks: 12185
Total 3D 

Stage 1 (linear): 100%|██████████| 50/50 [00:28<00:00] , best=1.123, iter=45, median=0.868px, rmse=1.123


Stage 1 (linear): 28.0s | nfev=50 njev=46 | The maximum number of function evaluations is exceeded.


Stage 2 (huber):  56%|█████▌    | 28/50 [00:00<00:00] , best=1.123, iter=4, median=0.868px, rmse=1.124  

Stage 2 (huber): 0.9s | nfev=28 njev=5 | Both `ftol` and `xtol` termination conditions are satisfied.
Final reprojection RMSE: 1.1239 | median err: 0.8676 px | inliers (<2px): 85.8%

Final reconstruction:
Cameras: 36
3D points: 3039


In [130]:

import numpy as np
import open3d as o3d

def visualize_point_cloud(points3d, point_size=3.0):
    xyz = []

    for p in points3d.values():
        if isinstance(p, dict):
            p = p["xyz"]
        xyz.append(np.asarray(p).reshape(3))

    xyz = np.asarray(xyz, dtype=float)
    xyz = xyz[np.isfinite(xyz).all(axis=1)]

    if len(xyz) == 0:
        raise ValueError("No valid 3D points found.")

    # Remove extreme outliers for clearer visualization
    median = np.median(xyz, axis=0)
    distance = np.linalg.norm(xyz - median, axis=1)
    limit = np.percentile(distance, 99)

    xyz = xyz[distance <= limit]

    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(xyz)

    # Color by height
    z = xyz[:, 2]
    z_norm = (z - z.min()) / (np.ptp(z) + 1e-12)
    colors = np.column_stack((
        z_norm,
        0.3 * np.ones(len(z)),
        1.0 - z_norm
    ))
    cloud.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries(
        [cloud],
        window_name="SfM Reconstruction",
        width=1200,
        height=800,
        point_show_normal=False
    )


In [131]:
visualize_point_cloud(points3d, point_size=3.0)

In [132]:
# Output directory
out_dir = Path("colmap_export")
out_dir.mkdir(exist_ok=True)

camera_id = 1
image_id_map = {cid: cid + 1 for cid in cameras}

# Map track IDs to COLMAP 3D point IDs
point_id_map = {tid: tid + 1 for tid in points3d if tid in tracks}

# cameras.txt
d = np.asarray(dist).ravel()

k1 = d[0] if len(d) > 0 else 0.0
k2 = d[1] if len(d) > 1 else 0.0
p1 = d[2] if len(d) > 2 else 0.0
p2 = d[3] if len(d) > 3 else 0.0

height, width = images[0].shape[:2]

with open(out_dir / "cameras.txt", "w") as f:
    f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
    f.write(
        f"{camera_id} OPENCV {width} {height} "
        f"{K[0,0]:.12g} {K[1,1]:.12g} "
        f"{K[0,2]:.12g} {K[1,2]:.12g} "
        f"{k1:.12g} {k2:.12g} {p1:.12g} {p2:.12g}\n"
    )

# Build each image's 2D observation
# Each observation is (feature_idx, x, y, point3D_id).
# We also need the observation index for points3D.txt. 

image_observations = {}
point_tracks = {}

for cid in sorted(cameras):
    observations = []

    for tid, track in tracks.items():
        if cid not in track:
            continue

        feature_idx = track[cid]

        if tid in points3d:
            pid = point_id_map.get(tid, -1)
        else:
            pid = -1

        # original SIFT feature location
        x, y = keypoints[cid][feature_idx].pt
        observations.append((feature_idx, x, y, pid, tid))

    # Deterministic order: feature index.
    observations.sort(key=lambda obs: obs[0])

    image_id = image_id_map[cid]
    image_observations[cid] = observations

    for point2D_idx, (_, x, y, pid, tid) in enumerate(observations):
        if pid != -1:
            point_tracks.setdefault(tid, []).append((image_id, point2D_idx))

# images.txt
with open(out_dir / "images.txt", "w") as f:
    f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
    f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")

    for cid in sorted(cameras):
        P = cameras[cid]

        # Your camera matrices are P = K [R | t].
        Rt = np.linalg.inv(K) @ P
        R = Rt[:, :3]
        t = Rt[:, 3]

        qx, qy, qz, qw = Rotation.from_matrix(R).as_quat()

        image_id = image_id_map[cid]
        name = os.path.basename(filenames[cid])

        f.write(
            f"{image_id} {qw:.12g} {qx:.12g} "
            f"{qy:.12g} {qz:.12g} "
            f"{t[0]:.12g} {t[1]:.12g} {t[2]:.12g} "
            f"{camera_id} {name}\n"
        )

        obs_strings = []
        for _, x, y, pid, _ in image_observations[cid]:
            obs_strings.extend([f"{x:.6f}", f"{y:.6f}", str(pid)])

        f.write(" ".join(obs_strings) + "\n")


# points3D.txt
with open(out_dir / "points3D.txt", "w") as f:
    f.write("# POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[]\n")

    for tid, xyz in points3d.items():
        if tid not in point_id_map:
            continue

        pid = point_id_map[tid]
        xyz = np.asarray(xyz).ravel()

        # Estimate point colour from its image observations.
        colors = []

        for cid, feature_idx in tracks[tid].items():
            if cid not in cameras:
                continue

            x, y = keypoints[cid][feature_idx].pt
            img = images[cid]

            px = int(round(x))
            py = int(round(y))

            if 0 <= px < img.shape[1] and 0 <= py < img.shape[0]:
                pixel = img[py, px]

                if img.ndim == 3:
                    colors.append(pixel[::-1])  # BGR -> RGB
                else:
                    colors.append([pixel, pixel, pixel])

        if colors:
            rgb = np.clip(np.mean(colors, axis=0), 0, 255).astype(int)
        else:
            rgb = np.array([128, 128, 128])

        # Compute mean reprojection error in pixels
        errors = []

        for cid, feature_idx in tracks[tid].items():
            if cid not in cameras:
                continue

            P = cameras[cid]
            Xh = np.append(xyz, 1.0)
            projected = P @ Xh

            if projected[2] <= 1e-9:
                continue

            predicted = projected[:2] / projected[2]
            observed = np.asarray(keypoints[cid][feature_idx].pt)
            errors.append(np.linalg.norm(predicted - observed))

        error = float(np.mean(errors)) if errors else 0.0

        track_entries = []
        for image_id, point2D_idx in point_tracks.get(tid, []):
            track_entries.extend([str(image_id), str(point2D_idx)])

        f.write(
            f"{pid} {xyz[0]:.12g} {xyz[1]:.12g} {xyz[2]:.12g} "
            f"{rgb[0]} {rgb[1]} {rgb[2]} {error:.6f} "
            f"{' '.join(track_entries)}\n"
        )

print("COLMAP model exported to:", out_dir.resolve())
print("Cameras:", len(cameras))
print("3D points:", len(point_id_map))

COLMAP model exported to: D:\3D_Construction\colmap_export
Cameras: 36
3D points: 3039
